In [ ]:
import glob, os, gc, numpy as np, pandas as pd, torch, torch.nn as nn

DATA = glob.glob("/kaggle/input/**/test_meta.parquet", recursive=True)[0].rsplit("/", 1)[0]
WEIGHTS = glob.glob("/kaggle/input/**/gnn_stream_b6.pt", recursive=True)[0]
print("DATA:", DATA)
print("WEIGHTS:", WEIGHTS)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
geometry = pd.read_csv(f"{DATA}/sensor_geometry.csv")
geo = geometry[["x", "y", "z"]].values.astype(np.float32) / 500.0

def build_features(batch_df, meta_df, maxlen=128):
    p_sensor = batch_df["sensor_id"].values
    p_time   = batch_df["time"].values.astype(np.float32)
    p_charge = batch_df["charge"].values.astype(np.float32)
    p_aux    = batch_df["auxiliary"].values.astype(np.float32)
    N = len(meta_df)
    X = np.zeros((N, maxlen, 6), dtype=np.float32)
    M = np.zeros((N, maxlen), dtype=bool)
    firsts, lasts = meta_df.first_pulse_index.values, meta_df.last_pulse_index.values
    for i in range(N):
        s, e = firsts[i], lasts[i] + 1
        order = np.argsort(p_aux[s:e] * 1e9 + p_time[s:e])[:maxlen]
        idx = np.arange(s, e)[order]
        n = len(idx)
        X[i, :n, 0:3] = geo[p_sensor[idx]]
        X[i, :n, 3]   = (p_time[idx] - p_time[idx].min()) / 1e4
        X[i, :n, 4]   = np.log10(p_charge[idx]) / 3.0
        X[i, :n, 5]   = p_aux[idx]
        M[i, :n] = True
    return X, M

def knn_indices(pos, mask, k):
    d = torch.cdist(pos, pos)
    d = d.masked_fill(~mask[:, None, :], float("inf"))
    return d.topk(k, dim=-1, largest=False).indices

def gather_neighbors(h, idx):
    b = torch.arange(h.shape[0], device=h.device)[:, None, None]
    return h[b, idx]

class EdgeConv(nn.Module):
    def __init__(self, c_in, c_out):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(2*c_in, c_out), nn.LayerNorm(c_out), nn.GELU(),
            nn.Linear(c_out, c_out), nn.GELU())
        self.skip = nn.Linear(c_in, c_out)
    def forward(self, h, idx, nbr_mask, mask):
        hj = gather_neighbors(h, idx)
        hi = h.unsqueeze(2).expand_as(hj)
        msg = self.mlp(torch.cat([hi, hj - hi], dim=-1))
        msg = msg.masked_fill(~nbr_mask.unsqueeze(-1), -1e9)
        out = msg.max(dim=2).values + self.skip(h)
        return out * mask.unsqueeze(-1)

class IceCubeGNN(nn.Module):
    def __init__(self, k=8, hidden=64, n_layers=3):
        super().__init__()
        self.k = k
        self.embed = nn.Sequential(nn.Linear(6, hidden), nn.GELU())
        self.layers = nn.ModuleList([EdgeConv(hidden, hidden) for _ in range(n_layers)])
        self.head = nn.Sequential(nn.Linear(2*hidden, 128), nn.GELU(), nn.Linear(128, 3))
    def forward(self, x, mask):
        idx = knn_indices(x[..., :4], mask, self.k)
        b = torch.arange(x.shape[0], device=x.device)[:, None, None]
        nbr_mask = mask[b, idx]
        h = self.embed(x) * mask.unsqueeze(-1)
        for layer in self.layers:
            h = layer(h, idx, nbr_mask, mask)
        m = mask.unsqueeze(-1)
        h_mean = (h * m).sum(1) / m.sum(1).clamp(min=1)
        h_max  = h.masked_fill(~m, -1e9).max(1).values
        out = self.head(torch.cat([h_mean, h_max], dim=-1))
        return out / out.norm(dim=-1, keepdim=True).clamp(min=1e-8)

#Trained weights
model = IceCubeGNN(k=8, hidden=128, n_layers=3).to(device)
model.load_state_dict(torch.load(WEIGHTS, map_location=device))
model.eval()

def vec_to_angles(v):
    zenith  = np.arccos(np.clip(v[:, 2], -1, 1))
    azimuth = np.arctan2(v[:, 1], v[:, 0]) % (2 * np.pi)
    return azimuth, zenith

#Testing prediction on every batch
test_meta = pd.read_parquet(f"{DATA}/test_meta.parquet")
rows = []
for b in test_meta.batch_id.unique():
    tm = test_meta[test_meta.batch_id == b].reset_index(drop=True)
    tb = pd.read_parquet(f"{DATA}/test/batch_{b}.parquet")
    Xt, Mt = build_features(tb, tm)
    Xt, Mt = torch.from_numpy(Xt).to(device), torch.from_numpy(Mt).to(device)
    preds = []
    with torch.no_grad():
        for i in range(0, len(Xt), 512):
            preds.append(model(Xt[i:i+512], Mt[i:i+512]).cpu().numpy())
    az, zen = vec_to_angles(np.concatenate(preds))
    rows.append(pd.DataFrame({"event_id": tm.event_id.values, "azimuth": az, "zenith": zen}))
    del tb, Xt, Mt; gc.collect()

sub = pd.concat(rows)
sub.to_csv("submission.csv", index=False)
print(sub.head(), len(sub))